# Olist RFM Customer Segmentation Analysis
## Author: Ahmed Mohamed Awadalla
## Date: May 2026

This notebook performs RFM (Recency, Frequency, Monetary) analysis to segment customers based on their purchasing behavior.

**RFM Definitions:**
- **Recency**: Days since last purchase (lower is better)
- **Frequency**: Number of orders (higher is better)
- **Monetary**: Total amount spent (higher is better)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Libraries loaded successfully")

## 2. Load Cleaned Data

In [ ]:
clean_data_path = "../data/cleaned/"

orders = pd.read_csv(clean_data_path + "orders_clean.csv")
customers = pd.read_csv(clean_data_path + "customers_clean.csv")
order_items = pd.read_csv(clean_data_path + "order_items_clean.csv")
payments = pd.read_csv(clean_data_path + "order_payments_clean.csv")

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

delivered_orders = orders[orders['order_status'] == 'delivered'].copy()

print(f"Delivered orders: {len(delivered_orders):,}")
print(f"Date range: {delivered_orders['order_purchase_timestamp'].min()} to {delivered_orders['order_purchase_timestamp'].max()}")

## 3. Calculate Monetary Value per Order

In [ ]:
order_items['total_value'] = order_items['price'] + order_items['freight_value']
order_value = order_items.groupby('order_id')['total_value'].sum().reset_index()
order_value.columns = ['order_id', 'order_total']

delivered_with_value = delivered_orders.merge(order_value, on='order_id', how='inner')
customer_orders = delivered_with_value.merge(customers, on='customer_id', how='inner')

print(f"Customer orders with value: {len(customer_orders):,}")
print(f"Total revenue: ${customer_orders['order_total'].sum():,.2f}")

## 4. Calculate RFM Metrics per Customer

In [ ]:
reference_date = customer_orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
print(f"Reference date for Recency calculation: {reference_date.date()}")

rfm = customer_orders.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - x.max()).days,
    'order_id': 'nunique',
    'order_total': 'sum'
}).reset_index()

rfm.columns = ['customer_unique_id', 'recency', 'frequency', 'monetary']

print(f"\nTotal customers analyzed: {len(rfm):,}")
print("\nRFM Statistics:")
print(f"  Recency - Min: {rfm['recency'].min()} days, Max: {rfm['recency'].max()} days, Mean: {rfm['recency'].mean():.1f} days")
print(f"  Frequency - Min: {rfm['frequency'].min()}, Max: {rfm['frequency'].max()}, Mean: {rfm['frequency'].mean():.2f}")
print(f"  Monetary - Min: ${rfm['monetary'].min():.2f}, Max: ${rfm['monetary'].max():.2f}, Mean: ${rfm['monetary'].mean():.2f}")

## 5. Create RFM Scores (1-4 Scale)

In [ ]:
def rfm_score_r(x):
    if x <= rfm['recency'].quantile(0.25):
        return 4
    elif x <= rfm['recency'].quantile(0.50):
        return 3
    elif x <= rfm['recency'].quantile(0.75):
        return 2
    else:
        return 1

def rfm_score_f(x):
    if x >= rfm['frequency'].quantile(0.75):
        return 4
    elif x >= rfm['frequency'].quantile(0.50):
        return 3
    elif x >= rfm['frequency'].quantile(0.25):
        return 2
    else:
        return 1

def rfm_score_m(x):
    if x >= rfm['monetary'].quantile(0.75):
        return 4
    elif x >= rfm['monetary'].quantile(0.50):
        return 3
    elif x >= rfm['monetary'].quantile(0.25):
        return 2
    else:
        return 1

rfm['R_score'] = rfm['recency'].apply(rfm_score_r)
rfm['F_score'] = rfm['frequency'].apply(rfm_score_f)
rfm['M_score'] = rfm['monetary'].apply(rfm_score_m)

rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print("RFM scores calculated")
print(rfm[['customer_unique_id', 'recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_total']].head())

print("\nScore Distribution:")
print("  R_score (Recency) - 4: Best, 1: Worst")
print("  F_score (Frequency) - 4: Best, 1: Worst")
print("  M_score (Monetary) - 4: Best, 1: Worst")

## 6. Customer Segmentation

In [ ]:
def customer_segment(row):
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return 'Champions'
    elif row['R_score'] >= 3 and row['F_score'] >= 3 and row['M_score'] >= 3:
        return 'Loyal Customers'
    elif row['R_score'] >= 4 and row['F_score'] <= 2:
        return 'New Customers'
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return 'At Risk'
    elif row['R_score'] <= 1 and row['F_score'] >= 1:
        return 'Lost'
    elif row['R_score'] >= 3 and row['F_score'] <= 2:
        return 'Promising'
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return 'Hibernating'
    else:
        return 'Other'

rfm['Segment'] = rfm.apply(customer_segment, axis=1)

segment_stats = rfm.groupby('Segment').agg({
    'customer_unique_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).round(2).reset_index()

segment_stats.columns = ['Segment', 'Customer_Count', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']
segment_stats['Percentage'] = (segment_stats['Customer_Count'] / len(rfm)) * 100
segment_stats = segment_stats.sort_values('Customer_Count', ascending=False)

print("=" * 70)
print("CUSTOMER SEGMENTATION SUMMARY")
print("=" * 70)
print(segment_stats.to_string(index=False))

segment_revenue = rfm.groupby('Segment')['monetary'].sum().sort_values(ascending=False)
print("\n" + "=" * 50)
print("REVENUE BY SEGMENT")
print("=" * 50)
for seg, rev in segment_revenue.items():
    pct = (rev / rfm['monetary'].sum()) * 100
    print(f"  {seg}: ${rev:,.2f} ({pct:.1f}%)")

## 7. Visualize Customer Segments

In [ ]:
plt.figure(figsize=(12, 8))
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#5D9B9B', '#A37C6E', '#4CAF50']
plt.pie(segment_stats['Customer_Count'], labels=segment_stats['Segment'], autopct='%1.1f%%',
        startangle=90, colors=colors[:len(segment_stats)])
plt.title('Customer Segment Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=segment_stats, x='Segment', y='Customer_Count', palette='Blues_d')
plt.title('Number of Customers by Segment', fontsize=14, fontweight='bold')
plt.xlabel('Segment', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=segment_stats, x='Segment', y='Avg_Monetary', palette='Greens_r')
plt.title('Average Monetary Value by Segment', fontsize=14, fontweight='bold')
plt.xlabel('Segment', fontsize=12)
plt.ylabel('Average Monetary ($)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. RFM Heatmap

In [ ]:
heatmap_data = segment_stats.set_index('Segment')[['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']]
heatmap_data_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data_norm, annot=heatmap_data, fmt='.1f', cmap='coolwarm', linewidths=1)
plt.title('RFM Metrics by Customer Segment (Normalized)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Pareto Principle (80/20 Rule)

In [ ]:
rfm_sorted = rfm.sort_values('monetary', ascending=False).reset_index(drop=True)
rfm_sorted['cumulative_revenue'] = rfm_sorted['monetary'].cumsum()
rfm_sorted['cumulative_revenue_pct'] = (rfm_sorted['cumulative_revenue'] / rfm_sorted['monetary'].sum()) * 100
rfm_sorted['cumulative_customers_pct'] = (rfm_sorted.index + 1) / len(rfm_sorted) * 100

pct_80_customers = rfm_sorted[rfm_sorted['cumulative_revenue_pct'] >= 80].iloc[0]['cumulative_customers_pct']

plt.figure(figsize=(12, 6))
plt.plot(rfm_sorted['cumulative_customers_pct'], rfm_sorted['cumulative_revenue_pct'], 
         linewidth=2, color='#2E86AB')
plt.axhline(y=80, color='red', linestyle='--', linewidth=1.5, label='80% Revenue')
plt.axvline(x=pct_80_customers, color='green', linestyle='--', linewidth=1.5, 
            label=f'{pct_80_customers:.1f}% of Customers')
plt.fill_between(rfm_sorted['cumulative_customers_pct'], 0, rfm_sorted['cumulative_revenue_pct'], 
                  alpha=0.3, color='#2E86AB')
plt.xlabel('Percentage of Customers', fontsize=12)
plt.ylabel('Percentage of Revenue', fontsize=12)
plt.title('Pareto Principle: Customer Revenue Concentration', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 50)
print("PARETO PRINCIPLE (80/20 RULE)")
print("=" * 50)
print(f"Top {pct_80_customers:.1f}% of customers generate 80% of revenue")
print(f"(This means {100 - pct_80_customers:.1f}% of customers generate only 20% of revenue)")

## 10. Marketing Recommendations

In [ ]:
recommendations = {
    'Champions': 'Offer VIP perks, loyalty programs, and exclusive previews.',
    'Loyal Customers': 'Provide cross-sell and upsell opportunities.',
    'Promising': 'Nurture with engagement campaigns.',
    'New Customers': 'Welcome series and first-repeat purchase incentives.',
    'At Risk': 'Re-engagement campaigns with special offers.',
    'Hibernating': 'Seasonal promotions or reactivation emails.',
    'Lost': 'Very low priority.',
    'Other': 'Monitor and collect more data.'
}

print("=" * 70)
print("MARKETING RECOMMENDATIONS BY SEGMENT")
print("=" * 70)

for segment in segment_stats['Segment']:
    count = segment_stats[segment_stats['Segment'] == segment]['Customer_Count'].values[0]
    pct = segment_stats[segment_stats['Segment'] == segment]['Percentage'].values[0]
    print(f"\n{segment}:")
    print(f"   Customers: {count:,} ({pct:.1f}%)")
    print(f"   Action: {recommendations.get(segment, 'No specific recommendation')}")

## 11. Export RFM Results

In [ ]:
rfm.to_csv("../data/cleaned/rfm_analysis.csv", index=False)
segment_stats.to_csv("../data/cleaned/rfm_segments.csv", index=False)
print("RFM analysis saved to: ../data/cleaned/rfm_analysis.csv")
print("Segment statistics saved to: ../data/cleaned/rfm_segments.csv")

## 12. Final Summary

In [ ]:
print("=" * 70)
print("RFM ANALYSIS - FINAL SUMMARY")
print("=" * 70)

print(f"\nTotal Customers Analyzed: {len(rfm):,}")
print(f"Total Revenue: ${rfm['monetary'].sum():,.2f}")
print(f"Average Revenue per Customer: ${rfm['monetary'].mean():.2f}")
print(f"Average Order Frequency: {rfm['frequency'].mean():.2f} orders")
print(f"Average Recency: {rfm['recency'].mean():.1f} days")

print("\n" + "=" * 70)
print("RFM ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 70)